# Indonesian Disaster News NER

This notebook runs a single NER scenario only: **pretrained inference without fine-tuning**.  
The final output is an article-level CSV with the same structure as `ner_results.csv`:

`id | LOCATION | ORGANIZATION | PERSON | DISASTER_TYPE | DATE/TIME`


In [ ]:
%pip -q install transformers pandas numpy torch tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import re
import gc
import warnings
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

warnings.filterwarnings("ignore")

# -------------------------
# Configuration
# -------------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Update this path if your CSV is stored elsewhere.
CSV_PATH = "/content/drive/MyDrive/Kebutuhan_NLP_X_DL_Kelompok_6/NLP/berita_bencana.csv"
OUTPUT_DIR = "ner_outputs_single_scenario"
OUTPUT_CSV_NAME = "ner_results.csv"

MODEL_NAME = "cahya/bert-base-indonesian-NER"

TITLE_COL = "judul_berita"
TEXT_COL = "full_text"
DATE_COL = "tanggal"
AUTHOR_COL = "author"

SEGMENT_MAX_CHARS = 280
SEGMENT_MIN_CHARS = 40
PIPELINE_CHUNK_TOKENS = 220
PIPELINE_STRIDE_TOKENS = 48

DEVICE = 0 if torch.cuda.is_available() else -1

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Model name:", MODEL_NAME)
print("CSV path:", CSV_PATH)
print("Output directory:", OUTPUT_DIR)

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Model name: cahya/bert-base-indonesian-NER
CSV path: /content/drive/MyDrive/Kebutuhan_NLP_X_DL_Kelompok_6/NLP/berita_bencana.csv
Output directory: ner_outputs_single_scenario


A small preparation step is used before inference.  
The notebook combines the title and article body, normalizes whitespace, and creates a clean `id` column for the final CSV.


In [ ]:
df = pd.read_csv(CSV_PATH).copy()

required_columns = [TITLE_COL, TEXT_COL, DATE_COL, AUTHOR_COL]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

def normalize_text(text: str) -> str:
    text = str(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Match the output style in ner_results.csv with a clean 0-based id.
df["id"] = np.arange(len(df))
df["title_text"] = df[TITLE_COL].fillna("").astype(str).map(normalize_text)
df["body_text"] = df[TEXT_COL].fillna("").astype(str).map(normalize_text)
df["published_at"] = df[DATE_COL].fillna("").astype(str)
df["author_name"] = df[AUTHOR_COL].fillna("").astype(str)

df["document_text"] = (
    df["title_text"].replace("", np.nan).fillna("") +
    np.where(df["title_text"].str.len() > 0, ". ", "") +
    df["body_text"].replace("", np.nan).fillna("")
).map(normalize_text)

display(df[["id", TITLE_COL, DATE_COL, AUTHOR_COL]].head())
print("Dataset shape:", df.shape)

,id,judul_berita,tanggal,author
0,0,Aceh Tetapkan Status Siaga Bencana hingga 20 A...,2026-04-14 14:19:49,Feri Agus
1,1,FAO Waspadai Bencana Pangan Global Imbas Gangg...,2026-04-14 11:18:38,Putri Utami
2,2,Mendagri Tito Sebut Inflasi Bulanan 3 Daerah T...,2026-04-06 15:52:22,Indra Hendriana
3,3,"Gajah Dikerahkan Bantu Bencana, BKSDA Pastikan...",2025-12-09 12:47:24,Yugo Hindarto
4,4,Ada 2.463 Titik Rawan Bencana di Indonesia Jel...,2025-12-19 13:03:57,Yugo Hindarto


Dataset shape: (2000, 11)


In [ ]:
def split_into_segments(text: str, max_chars: int = SEGMENT_MAX_CHARS, min_chars: int = SEGMENT_MIN_CHARS) -> List[str]:
    text = normalize_text(text)
    if not text:
        return []

    candidate_sentences = re.split(r"(?<=[\.!?])\s+|\n+", text)
    candidate_sentences = [s.strip() for s in candidate_sentences if s and s.strip()]

    segments = []
    current = ""

    for sentence in candidate_sentences:
        if len(sentence) > max_chars:
            if current:
                segments.append(current.strip())
                current = ""
            for start in range(0, len(sentence), max_chars):
                part = sentence[start:start + max_chars].strip()
                if part:
                    segments.append(part)
            continue

        if not current:
            current = sentence
        elif len(current) + 1 + len(sentence) <= max_chars:
            current = f"{current} {sentence}"
        else:
            segments.append(current.strip())
            current = sentence

    if current:
        segments.append(current.strip())

    merged = []
    buffer_text = ""

    for seg in segments:
        if len(seg) < min_chars:
            if buffer_text:
                buffer_text = f"{buffer_text} {seg}".strip()
            else:
                buffer_text = seg
        else:
            if buffer_text:
                merged.append(f"{buffer_text} {seg}".strip())
                buffer_text = ""
            else:
                merged.append(seg)

    if buffer_text:
        merged.append(buffer_text.strip())

    return [seg for seg in merged if seg]

segment_rows = []
for _, row in df.iterrows():
    segments = split_into_segments(row["document_text"])
    if not segments:
        segments = [""]
    for segment_idx, segment_text in enumerate(segments):
        segment_rows.append(
            {
                "id": row["id"],
                "segment_id": segment_idx,
                "text": segment_text,
            }
        )

segments_df = pd.DataFrame(segment_rows)
segments_df["segment_length"] = segments_df["text"].fillna("").str.len()

display(segments_df.head(10))
print("Number of segments:", len(segments_df))
print("Average segment length:", round(segments_df["segment_length"].mean(), 2))

,id,segment_id,text,segment_length
0,0,0,Aceh Tetapkan Status Siaga Bencana hingga 20 A...,165
1,0,1,Kebijakan ini diambil menyusul peringatan dini...,119
2,0,2,Sebelumnya BMKG memperkirakan wilayah Aceh ber...,169
3,0,3,"Kondisi ini dipengaruhi oleh pola siklonik, be...",118
4,0,4,Dampaknya hampir seluruh wilayah Aceh berisiko...,204
5,0,5,Nasir menginstruksikan seluruh pemerintah kabu...,274
6,0,6,Periode ini sangat krusial untuk meminimalisir...,151
7,0,7,Pemerintah daerah diminta segera melakukan nor...,167
8,0,8,Selain itu petugas lapangan diminta meningkatk...,233
9,0,9,"""Sarana pendukung seperti perahu motor, kendar...",160


Number of segments: 25557
Average segment length: 205.56


The model is used through the Hugging Face **token-classification pipeline**, which is one of the standard inference APIs for NER, and the final article-level table is exported with `DataFrame.to_csv`. citeturn286174search0turn286174search1

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipe = pipeline(
    task="token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=DEVICE,
)

def normalize_pretrained_label(raw_label: str):
    if raw_label is None:
        return None

    label = str(raw_label).upper().replace("B-", "").replace("I-", "")
    mapping = {
        "PER": "PERSON",
        "PERSON": "PERSON",
        "LOC": "LOCATION",
        "LOCATION": "LOCATION",
        "PLACE": "LOCATION",
        "GPE": "LOCATION",
        "ORG": "ORGANIZATION",
        "ORGANIZATION": "ORGANIZATION",
        "ORGANISATION": "ORGANIZATION",
        "DATE": "DATE/TIME",
        "TIME": "DATE/TIME",
        "DATETIME": "DATE/TIME",
        "DATE/TIME": "DATE/TIME",
        "DISASTER": "DISASTER_TYPE",
        "DISASTER_TYPE": "DISASTER_TYPE",
    }
    return mapping.get(label)

DISASTER_KEYWORDS = [
    "banjir", "banjir bandang", "gempa", "gempa bumi", "tanah longsor", "longsor",
    "puting beliung", "angin kencang", "kebakaran hutan", "karhutla", "letusan gunung",
    "tsunami", "kekeringan", "abrasi", "erupsi", "cuaca ekstrem",
    "bencana hidrometeorologi", "gelombang tinggi"
]

DATE_PATTERNS = [
    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
    r"\b\d{1,2}\s+(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\s+\d{2,4}\b",
    r"\b(senin|selasa|rabu|kamis|jumat|sabtu|minggu)\b",
    r"\bpukul\s+\d{1,2}[\.:]\d{2}\s*(wib|wita|wit)?\b",
    r"\b\d{1,2}:\d{2}\s*(wib|wita|wit)?\b",
]

def collect_rule_entities(text: str) -> List[Dict[str, Any]]:
    text_lower = text.lower()
    rule_entities = []

    for keyword in DISASTER_KEYWORDS:
        for match in re.finditer(re.escape(keyword.lower()), text_lower):
            rule_entities.append(
                {
                    "text": text[match.start():match.end()],
                    "start": match.start(),
                    "end": match.end(),
                    "label": "DISASTER_TYPE",
                    "score": 1.0,
                    "source": "rule_disaster",
                }
            )

    for pattern in DATE_PATTERNS:
        for match in re.finditer(pattern, text_lower, flags=re.IGNORECASE):
            rule_entities.append(
                {
                    "text": text[match.start():match.end()],
                    "start": match.start(),
                    "end": match.end(),
                    "label": "DATE/TIME",
                    "score": 1.0,
                    "source": "rule_datetime",
                }
            )

    return rule_entities

def chunk_text_for_ner(text: str, tokenizer, max_tokens: int = PIPELINE_CHUNK_TOKENS, stride: int = PIPELINE_STRIDE_TOKENS):
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )

    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    if len(input_ids) <= max_tokens:
        return [{"text": text, "char_start": 0, "char_end": len(text)}]

    chunks = []
    step = max_tokens - stride
    start_idx = 0

    while start_idx < len(input_ids):
        end_idx = min(start_idx + max_tokens, len(input_ids))
        char_start = offsets[start_idx][0]
        char_end = offsets[end_idx - 1][1]

        chunks.append(
            {
                "text": text[char_start:char_end],
                "char_start": char_start,
                "char_end": char_end,
            }
        )

        if end_idx >= len(input_ids):
            break

        start_idx += step

    return chunks

def resolve_overlaps(entities: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if not entities:
        return []

    entities = sorted(
        entities,
        key=lambda x: (x["start"], -(x["end"] - x["start"]), -float(x.get("score", 0.0))),
    )

    filtered = []
    occupied = []

    for ent in entities:
        overlap = False
        for start, end in occupied:
            if not (ent["end"] <= start or ent["start"] >= end):
                overlap = True
                break
        if not overlap:
            filtered.append(ent)
            occupied.append((ent["start"], ent["end"]))

    return sorted(filtered, key=lambda x: (x["start"], x["end"]))

def extract_entities_from_single_text(text: str, ner_pipe, tokenizer, add_rules: bool = True):
    chunks = chunk_text_for_ner(text=text, tokenizer=tokenizer)
    merged_entities = []

    for chunk in chunks:
        predictions = ner_pipe(chunk["text"])
        for pred in predictions:
            raw_label = pred.get("entity_group", pred.get("entity"))
            normalized_label = normalize_pretrained_label(raw_label)
            if normalized_label is None:
                continue

            global_start = int(pred["start"]) + chunk["char_start"]
            global_end = int(pred["end"]) + chunk["char_start"]

            merged_entities.append(
                {
                    "text": text[global_start:global_end],
                    "start": global_start,
                    "end": global_end,
                    "label": normalized_label,
                    "score": float(pred.get("score", 0.0)),
                    "source": "pretrained_model",
                }
            )

    if add_rules:
        merged_entities.extend(collect_rule_entities(text))

    return resolve_overlaps(merged_entities)

def unique_join(values):
    seen = set()
    ordered = []
    for value in values:
        value = str(value).strip().lower()
        if not value or value in seen:
            continue
        seen.add(value)
        ordered.append(value)
    return ", ".join(ordered)

def build_article_level_output(df_articles: pd.DataFrame, segments_df: pd.DataFrame, ner_pipe, tokenizer):
    article_entities = []

    for row in tqdm(segments_df.to_dict(orient="records"), total=len(segments_df), desc="Running NER"):
        segment_entities = extract_entities_from_single_text(
            text=row["text"],
            ner_pipe=ner_pipe,
            tokenizer=tokenizer,
            add_rules=True,
        )

        for ent in segment_entities:
            article_entities.append(
                {
                    "id": row["id"],
                    "label": ent["label"],
                    "entity_text": ent["text"],
                    "segment_id": row["segment_id"],
                }
            )

    entities_df = pd.DataFrame(article_entities)

    final_columns = ["id", "LOCATION", "ORGANIZATION", "PERSON", "DISASTER_TYPE", "DATE/TIME"]

    if entities_df.empty:
        result_df = df_articles[["id"]].copy()
        for col in final_columns[1:]:
            result_df[col] = ""
        return result_df[final_columns], entities_df

    entities_df = entities_df[entities_df["label"].isin(final_columns[1:])].copy()
    entities_df = entities_df.sort_values(["id", "label", "segment_id"]).reset_index(drop=True)

    grouped = (
        entities_df
        .groupby(["id", "label"], as_index=False)["entity_text"]
        .agg(unique_join)
    )

    wide = (
        grouped
        .pivot(index="id", columns="label", values="entity_text")
        .reset_index()
    )

    wide.columns.name = None

    result_df = df_articles[["id"]].merge(wide, on="id", how="left")

    for col in final_columns[1:]:
        if col not in result_df.columns:
            result_df[col] = ""
        result_df[col] = result_df[col].fillna("")

    result_df = result_df[final_columns].copy()
    return result_df, entities_df

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
ner_results_df, segment_entities_df = build_article_level_output(
    df_articles=df,
    segments_df=segments_df,
    ner_pipe=ner_pipe,
    tokenizer=tokenizer,
)

display(ner_results_df.head(20))
print("Output shape:", ner_results_df.shape)

Running NER:   0%|          | 0/25557 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,id,LOCATION,ORGANIZATION,PERSON,DISASTER_TYPE,DATE/TIME
0,0,aceh,"badan meteorologi, klimatologi, dan geofisika,...","m, nasir","bencana hidrometeorologi, cuaca ekstrem, angin...","20 april 2026, selasa"
1,1,selat hormuz,"fao, organisasi pangan dan pertanian, anadolu ...","maximo torero, torero, gamb, o, david laborde,...",,
2,2,"aceh, sumatra barat, sumatra utara, kantor, ja...","pusat k, badan pengawas obat, bp, badan pusat ...","tito, muhammad tito karnavian, taruna ikrar, a...",,senin
3,3,"kabupaten pidie jaya, asia, indonesia","konservasi sumber daya alam, ksda aceh","ujang wisnu barata, ujang","banjir, tsunami",selasa
4,4,"indonesia, nata, pulau, sumatra, sulawesi, pul...","ru, antara, badan meteorologi, klimatologi, da...","komisaris jenderal polisi dedi prasetyo, dedi","banjir, longsor","senin, 29 desember 2025, 10 januari 2026"
5,5,"sumatra, provinsi aceh, sumatra barat, sumatra...","otoritas jasa keuangan, ojk, dewan komisioner ...","mahendra siregar, mahendra","banjir, longsor",10 desember 2025
6,6,"istana kepresidenan, jakarta pusat, provinsi a...","pt sarana multi infrastruktur (persero), pt sm...","purba, purbaya yudhi sadewa, purbaya","banjir bandang, longsor, banjir","senin, minggu"
7,7,"sumatra, sumatra barat, sumatra utara, aceh, s...","bni, pt bank negara indonesia (persero) tbk, c...","okki rushartomo, okki","banjir bandang, longsor, banjir","rabu, 29 november 2025, 30 november 2025"
8,8,"kepulauan sitaro, kabupaten kepulauan siau, ta...",pengendalian operasi bn,"abdul muhari, n, abdul","banjir bandang, bencana hidrometeorologi","rabu, 18 januari 2026, selasa, pukul 14.00 wib..."
9,9,"cirebon, jawa barat, kabupaten kuningan, sunga...","bd, tim pusat pengendalian operasi, pusdalops","hadi eko, eko",banjir,"selasa, rabu"


Output shape: (2000, 6)


In [ ]:
output_csv_path = os.path.join(OUTPUT_DIR, OUTPUT_CSV_NAME)

ner_results_df.to_csv(output_csv_path, index=False)

print("Saved final CSV to:", output_csv_path)

# Optional: also save the long-form segment-level extraction result for debugging
debug_csv_path = os.path.join(OUTPUT_DIR, "segment_level_ner_debug.csv")
segment_entities_df.to_csv(debug_csv_path, index=False)
print("Saved debug CSV to:", debug_csv_path)

Saved final CSV to: ner_outputs_single_scenario/ner_results.csv
Saved debug CSV to: ner_outputs_single_scenario/segment_level_ner_debug.csv
